In [1]:
from mask import create_masked_phi
from datasets import load_from_disk

In [5]:
from torch.optim.lr_scheduler import LinearLR
from torch.optim import SGD
from torch import nn
import torch

In [6]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        # Define layers
        self.hidden = nn.Linear(10, 5)  # Input layer (10 features) to hidden layer (5 neurons)
        self.output = nn.Linear(5, 1)   # Hidden layer to output layer (1 output)

    def forward(self, x):
        # Define forward pass
        x = torch.relu(self.hidden(x))  # Apply ReLU to hidden layer
        x = torch.sigmoid(self.output(x))  # Apply Sigmoid to output layer
        return x

# Initialize the network, define loss function and optimizer
net = SimpleNN()

In [7]:
lr = 0.2
optimizer = SGD(net.parameters(), lr=lr)

In [13]:
optimizer.__getstate__()

{'defaults': {'lr': 0.01,
  'momentum': 0,
  'dampening': 0,
  'weight_decay': 0,
  'nesterov': False,
  'maximize': False,
  'foreach': None,
  'differentiable': False,
  'fused': None},
 'state': defaultdict(dict, {}),
 'param_groups': [{'params': [Parameter containing:
    tensor([[ 0.1674,  0.0940, -0.3121,  0.0421,  0.1117, -0.2663,  0.1552, -0.0609,
             -0.0856, -0.2236],
            [-0.2676,  0.0637,  0.1466, -0.1549,  0.0362,  0.1487,  0.0148, -0.0449,
             -0.1742,  0.1151],
            [-0.2973,  0.2130, -0.2252, -0.1109, -0.0934,  0.2289,  0.0119, -0.1738,
              0.0596,  0.0299],
            [-0.0999,  0.0700,  0.1311,  0.1934,  0.1249,  0.1133, -0.3092,  0.1912,
             -0.2253, -0.0066],
            [ 0.2123, -0.0308,  0.1263, -0.0699, -0.0951,  0.0426,  0.2483, -0.0478,
              0.1458, -0.0985]], requires_grad=True),
    Parameter containing:
    tensor([ 0.0122,  0.1236, -0.1646,  0.1210,  0.0665], requires_grad=True),
    Parameter c

In [ ]:
warmup_percent = 0.1

In [ ]:
optimizer_steps = 1000

In [ ]:
target_lr = 1e-10

In [ ]:
initial_factor = lr/target_lr

In [ ]:
warmup_steps = int(optimizer_steps * warmup_percent)
initial_lr = 

In [9]:
# Assuming optimizer uses lr = 0.05 for all groups
# lr = 0.025    if epoch == 0
# lr = 0.03125  if epoch == 1
# lr = 0.0375   if epoch == 2
# lr = 0.04375  if epoch == 3
# lr = 0.05    if epoch >= 4

scheduler = LinearLR(optimizer, start_factor=target_lr, total_iters=warmup_steps)
for epoch in range(10):
    print('Epoch:', epoch, "LR", scheduler.get_last_lr())
    scheduler.step()

Epoch: 0 LR [1e-05]
Epoch: 1 LR [0.0025075]
Epoch: 2 LR [0.005005000000000001]
Epoch: 3 LR [0.0075025000000000005]
Epoch: 4 LR [0.010000000000000002]
Epoch: 5 LR [0.010000000000000002]
Epoch: 6 LR [0.010000000000000002]
Epoch: 7 LR [0.010000000000000002]
Epoch: 8 LR [0.010000000000000002]
Epoch: 9 LR [0.010000000000000002]


In [10]:
0.0075025000000000005/0.005005000000000001



1.499000999000999

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="auto",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
            ) 
tokenizer = AutoTokenizer.from_pretrained(model_name) 

In [3]:
target_layers = list(range(24,31))

In [ ]:
dataset = load_from_disk("./data/preprocessed/general_generated")
data = dataset[0]
data

In [ ]:
target_layers

In [6]:
role_map = {"maint_kg": 0, 
            "maint_lm": 1,
            "code_lm":1,
            "target_kg":2,}

tokenized_data = tokenizer.apply_chat_template([{"role": "user", "content": data["user"]}, {"role": "assistant", "content": data["assistant"]}], tokenize=True, padding="max_length", max_length=4096, truncation=True, return_tensors="pt")[0]
role = torch.tensor(role_map[data["role"]])


In [7]:
model = create_masked_phi(model, target_layers)

In [8]:
model.train();

In [ ]:
tokenized_data

In [ ]:
with torch.no_grad():
    output = model(input_ids=tokenized_data.to(model.device).unsqueeze(0), output_attentions=True)

In [ ]:
attentions = torch.cat(output.attentions).cpu()
attentions.shape

In [ ]:
output.logits